# SafeRL-Drive Phase-2 Colab driver

This notebook closes Phase 1 and runs the approved Phase-2 direct-versus-curriculum SAC study. GitHub and `/content/safedrive` hold live code; `/content/drive/MyDrive/SafeDrive` holds persistent artifacts. Each long curriculum stage is copied to Drive before the next begins. Google Drive mounting is sufficient, so PyDrive is not required.

Run sections in order. Confirmation seeds are deliberately blocked unless the seed-0 pilots meet the preregistered promising-result gate.

## 1. Initialize paths and experiment constants

In [3]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/djdhillxn/safedrive.git"
REPO_DIR = Path("/content/safedrive")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive" / "SafeDrive"
PHASE1_TEST_EPISODES = 20
PHASE2_TIMESTEPS = 500_000
PHASE2_TEST_EPISODES = 100
VIDEO_STEPS = 1_000
CONFIRMATION_SEEDS = [1, 2]

print(f"Live checkout: {REPO_DIR}")
print(f"Persistent artifacts: {DRIVE_PROJECT}")
print(f"Phase-2 maximum per condition: {PHASE2_TIMESTEPS:,} steps")

Live checkout: /content/safedrive
Persistent artifacts: /content/drive/MyDrive/SafeDrive
Phase-2 maximum per condition: 500,000 steps


In [4]:
print("hello")

hello


## 2. Runtime and GPU check

In [5]:
import torch

print(f"Python: {sys.version.split()[0]}")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=False)
else:
    print("nvidia-smi: unavailable")
print(f"PyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Python: 3.12.13
PyTorch CUDA: True
NVIDIA L4


## 3. Mount Google Drive

In [6]:
from google.colab import drive

drive.mount(str(DRIVE_MOUNT))
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
connection_file = DRIVE_PROJECT / "colab_connection_test.txt"
connection_file.write_text(f"Connected at {datetime.now(timezone.utc).isoformat()}\n", encoding="utf-8")
print(f"Drive read/write ready: {connection_file}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive read/write ready: /content/drive/MyDrive/SafeDrive/colab_connection_test.txt


## 4. Clone or fast-forward pull the public repository

In [7]:
if (REPO_DIR / ".git").exists():
    git_command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    if REPO_DIR.exists():
        REPO_DIR.rmdir()
    git_command = ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)]
git_result = subprocess.run(git_command, capture_output=True, text=True)
if git_result.returncode:
    print(git_result.stdout[-4000:])
    print(git_result.stderr[-4000:])
    raise RuntimeError("Git clone/pull failed.")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
git_commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(f"Repository ready at commit {git_commit[:12]}")

Repository ready at commit 2633492b1a79


## 4.1 Restore persistent artifacts from Drive

Full restore is intentional in Colab because evaluation, video, and curriculum resume need model files. The default Mac sync remains analysis-only.

In [8]:
subprocess.run([sys.executable, "-m", "scripts.sync_drive_runs", "--drive-project", str(DRIVE_PROJECT), "--local-runs", str(REPO_DIR / "runs"), "--include-training-artifacts"], cwd=REPO_DIR, check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'scripts.sync_drive_runs', '--drive-project', '/content/drive/MyDrive/SafeDrive', '--local-runs', '/content/safedrive/runs', '--include-training-artifacts'], returncode=0)

## 5. Install SafeRL-Drive

In [9]:
import importlib.metadata

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "gym"], check=False, capture_output=True, text=True)
install_result = subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--progress-bar", "off", "-e", "."], cwd=REPO_DIR, capture_output=True, text=True)
if install_result.returncode:
    print(install_result.stdout[-8000:])
    print(install_result.stderr[-8000:])
    raise RuntimeError("SafeRL-Drive installation failed.")
packages = ["metadrive-simulator", "stable-baselines3", "gymnasium", "torch"]
print(json.dumps({name: importlib.metadata.version(name) for name in packages}, indent=2))

{
  "metadrive-simulator": "0.4.3",
  "stable-baselines3": "2.9.0",
  "gymnasium": "1.3.0",
  "torch": "2.11.0+cu128"
}


### Shared artifact, gate, and restart-safe resume helpers

Run this cell after Drive restoration in every fresh kernel. It recreates the Phase-2 run paths, held-out summaries, `PILOTS_PROMISING`, and the list of complete paired seeds from saved pointers. The experiment cells below reuse completed work and advance only genuinely paused curriculum stages.

In [10]:
def read_latest_run(name):
    pointer = REPO_DIR / "runs" / f"latest_{name}.txt"
    if not pointer.exists():
        raise FileNotFoundError(f"Latest-run pointer not found: {pointer}")
    run_dir = Path(pointer.read_text(encoding="utf-8").strip())
    if not run_dir.is_absolute():
        run_dir = REPO_DIR / run_dir
    return run_dir.resolve()

def newest_run_containing(text):
    candidates = [path for path in (REPO_DIR / "runs").iterdir() if path.is_dir() and text in path.name]
    if not candidates:
        raise FileNotFoundError(f"No run directory contains {text!r}.")
    return max(candidates, key=lambda path: path.name)

def require_shell_success(label):
    exit_code = get_ipython().user_ns.get("_exit_code", 0)
    if exit_code:
        raise RuntimeError(f"{label} failed with exit code {exit_code}.")

def show_json(path):
    path = Path(path)
    data = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(data, indent=2))
    return data

def copy_to_drive(path, category="runs"):
    path = Path(path)
    target = DRIVE_PROJECT / category / path.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if path.is_dir():
        target.mkdir(parents=True, exist_ok=True)
        if shutil.which("rsync"):
            subprocess.run(["rsync", "-a", f"{path}/", f"{target}/"], check=True)
        else:
            shutil.copytree(path, target, dirs_exist_ok=True)
    else:
        shutil.copy2(path, target)
    print(f"Copied to Drive: {target}")
    return target

def backup_pointer_run(name):
    run_dir = read_latest_run(name)
    copy_to_drive(run_dir)
    copy_to_drive(REPO_DIR / "runs" / f"latest_{name}.txt")
    return run_dir

def require_metrics(summary, label, success=0.80, route=0.90, failures=0.10):
    values = {
        "success": float(summary.get("success_rate", 0.0)),
        "route": float(summary.get("mean_route_completion", 0.0)),
        "collision": float(summary.get("collision_rate", 1.0)),
        "off_road": float(summary.get("out_of_road_rate", 1.0)),
        "timeout": float(summary.get("timeout_or_max_step_rate", 1.0)),
    }
    passed = values["success"] >= success and values["route"] >= route and max(values["collision"], values["off_road"], values["timeout"]) <= failures
    print(label, {key: f"{value:.1%}" for key, value in values.items()}, "PASS" if passed else "FAIL")
    return passed

def require_curriculum_state(run_dir, expected):
    state = show_json(Path(run_dir) / "curriculum_state.json")
    if state.get("status") not in expected:
        raise RuntimeError(f"Curriculum status {state.get('status')!r}; expected {expected}.")
    return state

def representative_seed(run_dir):
    import pandas as pd
    frame = pd.read_csv(Path(run_dir) / "eval" / "best_test_episodes.csv")
    success_mask = frame["success"].astype(str).str.lower().eq("true")
    successful = frame[success_mask]
    if not successful.empty:
        return int(successful.iloc[0]["env_seed"])
    best = frame.loc[frame["route_completion"].astype(float).idxmax()]
    print(f"No successful rollout in {run_dir}; recording the highest-completion failure.")
    return int(best["env_seed"])

def read_json_file(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def optional_latest_run(name):
    try:
        return read_latest_run(name)
    except FileNotFoundError:
        return None

def learned_run_status(run_dir):
    if run_dir is None:
        return "missing"
    curriculum_state = Path(run_dir) / "curriculum_state.json"
    metadata = Path(run_dir) / "run_metadata.json"
    if curriculum_state.exists():
        return str(read_json_file(curriculum_state).get("status", "unknown"))
    if metadata.exists():
        return str(read_json_file(metadata).get("status", "unknown"))
    return "unknown"

def saved_summary(run_dir, prefix="best_test"):
    if run_dir is None:
        return None
    path = Path(run_dir) / "eval" / f"{prefix}_summary.json"
    return read_json_file(path) if path.exists() else None

def restore_phase2_session_state(quiet=False):
    records = []
    for seed in [0, 1, 2]:
        for condition, pointer, variable in [
            ("direct", f"sac_phase2_direct_seed{seed}", f"DIRECT_SEED{seed}"),
            ("curriculum", f"sac_phase2_curriculum_seed{seed}", f"CURRICULUM_SEED{seed}"),
        ]:
            run_dir = optional_latest_run(pointer)
            run_name = f"{variable}_RUN"
            summary_name = f"{variable}_SUMMARY"
            if run_dir is None:
                globals().pop(run_name, None)
                globals().pop(summary_name, None)
                records.append((seed, condition, "missing", False))
                continue
            globals()[run_name] = run_dir
            summary = saved_summary(run_dir)
            if summary is None:
                globals().pop(summary_name, None)
            else:
                globals()[summary_name] = summary
            records.append((seed, condition, learned_run_status(run_dir), summary is not None))

    direct_pilot = globals().get("DIRECT_SEED0_SUMMARY")
    curriculum_pilot = globals().get("CURRICULUM_SEED0_SUMMARY")
    if direct_pilot is not None and curriculum_pilot is not None:
        globals()["PILOTS_PROMISING"] = any(
            float(summary.get("success_rate", 0.0)) >= 0.50
            or float(summary.get("mean_route_completion", 0.0)) >= 0.80
            for summary in [direct_pilot, curriculum_pilot]
        )
    else:
        globals()["PILOTS_PROMISING"] = None

    complete_pairs = []
    for seed in [0, 1, 2]:
        direct_run = globals().get(f"DIRECT_SEED{seed}_RUN")
        curriculum_run = globals().get(f"CURRICULUM_SEED{seed}_RUN")
        if (
            learned_run_status(direct_run) == "complete"
            and learned_run_status(curriculum_run) == "complete"
            and saved_summary(direct_run) is not None
            and saved_summary(curriculum_run) is not None
        ):
            complete_pairs.append(seed)
    globals()["COMPLETED_PHASE2_SEEDS"] = complete_pairs
    globals()["COMPARISON_SEEDS"] = complete_pairs

    if not quiet:
        print("Restored Phase-2 notebook state from persistent run pointers:")
        for seed, condition, status, has_test in records:
            print(f"  seed {seed} {condition:10s}: {status:11s} | held-out test: {has_test}")
        print(f"PILOTS_PROMISING = {PILOTS_PROMISING}")
        print(f"Complete direct/curriculum seed pairs = {COMPLETED_PHASE2_SEEDS}")
    return records

def run_project_module(module, arguments, label):
    print(f"Starting {label}.")
    subprocess.run([sys.executable, "-m", module, *arguments], cwd=REPO_DIR, check=True)

def ensure_evaluation(run_dir, prefix="best_test", overrides=None):
    summary = saved_summary(run_dir, prefix=prefix)
    if summary is not None:
        print(f"Reusing saved evaluation: {Path(run_dir) / 'eval' / f'{prefix}_summary.json'}")
        return summary
    arguments = [
        "--run-dir", str(run_dir), "--model", "best", "--split", "test",
        "--episodes", str(PHASE2_TEST_EPISODES), "--prefix", prefix,
    ]
    arguments.extend(overrides or [])
    run_project_module("scripts.evaluate", arguments, f"{Path(run_dir).name} {prefix}")
    copy_to_drive(run_dir)
    return saved_summary(run_dir, prefix=prefix)

def ensure_direct_sac(seed):
    pointer = f"sac_phase2_direct_seed{seed}"
    run_dir = optional_latest_run(pointer)
    status = learned_run_status(run_dir)
    if status == "complete":
        print(f"Reusing complete seed-{seed} direct SAC run: {run_dir}")
    elif run_dir is None:
        run_project_module(
            "scripts.train",
            ["--config", "configs/sac_phase2_direct.yaml", "--seed", str(seed), "--total-timesteps", str(PHASE2_TIMESTEPS)],
            f"seed-{seed} direct SAC training",
        )
        run_dir = backup_pointer_run(pointer)
    else:
        raise RuntimeError(f"Direct seed-{seed} run is {status!r} and has no safe automatic resume: {run_dir}")
    ensure_evaluation(run_dir)
    restore_phase2_session_state(quiet=True)
    return run_dir

def advance_curriculum_sac(seed, target_stage):
    stage_targets = {"curve": 1, "straight_curve": 2, "procedural": 3}
    if target_stage not in stage_targets:
        raise ValueError(f"Unknown curriculum target stage: {target_stage}")
    target_index = stage_targets[target_stage]
    pointer = f"sac_phase2_curriculum_seed{seed}"
    run_dir = optional_latest_run(pointer)
    if run_dir is None:
        run_project_module(
            "scripts.train_curriculum",
            ["--config", "configs/sac_phase2_curriculum.yaml", "--seed", str(seed), "--stop-after-stage", "curve"],
            f"seed-{seed} curriculum curve stage",
        )
        run_dir = backup_pointer_run(pointer)

    while True:
        state = read_json_file(Path(run_dir) / "curriculum_state.json")
        status = state.get("status")
        next_index = int(state.get("next_stage_index", 0))
        if status == "failed_gate":
            last_stage = state.get("completed_stages", [{}])[-1]
            print(
                f"Seed-{seed} curriculum stopped at its required {last_stage.get('name')} gate; "
                "this is a terminal experimental outcome, not a resumable pause."
            )
            break
        if status == "complete" or next_index >= target_index:
            print(f"Reusing seed-{seed} curriculum through {target_stage}: {run_dir}")
            break
        if status != "paused":
            raise RuntimeError(f"Curriculum seed-{seed} is {status!r}; inspect {run_dir}")
        stage_name = state["stage_order"][next_index]
        arguments = [
            "--config", "configs/sac_phase2_curriculum.yaml", "--run-dir", str(run_dir), "--seed", str(seed),
        ]
        if stage_name != "procedural":
            arguments.extend(["--stop-after-stage", stage_name])
        run_project_module("scripts.train_curriculum", arguments, f"seed-{seed} curriculum {stage_name} stage")
        copy_to_drive(run_dir)

    if learned_run_status(run_dir) == "complete":
        ensure_evaluation(run_dir)
    restore_phase2_session_state(quiet=True)
    return run_dir

def ensure_baseline(policy, pointer, prefix):
    run_dir = optional_latest_run(pointer)
    summary_path = Path(run_dir) / "eval" / f"{prefix}_summary.json" if run_dir else None
    if summary_path is not None and summary_path.exists():
        print(f"Reusing saved {policy} baseline: {run_dir}")
        return run_dir
    run_project_module(
        "scripts.evaluate_baseline",
        ["--config", "configs/sac_phase2_direct.yaml", "--policy", policy, "--split", "test", "--episodes", str(PHASE2_TEST_EPISODES), "--prefix", prefix],
        f"Phase-2 {policy} test baseline",
    )
    return backup_pointer_run(pointer)

def ensure_video(run_dir, seed):
    existing = list((Path(run_dir) / "videos").glob(f"*_best_seed{seed}_topdown.mp4"))
    if existing:
        print(f"Reusing saved video: {existing[0]}")
        return existing[0]
    run_project_module(
        "scripts.record_video",
        ["--run-dir", str(run_dir), "--model", "best", "--seed", str(seed), "--steps", str(VIDEO_STEPS)],
        f"video for {Path(run_dir).name}",
    )
    copy_to_drive(run_dir)

PHASE2_RESTORED_STATE = restore_phase2_session_state()

## 6. Lightweight wiring test

In [11]:
!python -m scripts.train --config configs/smoke_test.yaml
require_shell_success("Smoke test")
SMOKE_RUN_DIR = backup_pointer_run("smoke")
print(SMOKE_RUN_DIR)

2026-07-22 15:22:17.273213: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 15:22:17.346083: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
INFO: Run directory: runs/20260722_152221_smoke_test_ppo_seed0
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: Fu

## 7. Close Phase 1 with the exact IDM held-out test

This is intentionally different from the already-passing 10-episode validation reproducibility gate. The comparison command will refuse any stale incompatible baseline.

In [10]:
!python -m scripts.evaluate_baseline --config configs/ppo_mvp.yaml --policy idm --split test --episodes {PHASE1_TEST_EPISODES} --prefix idm_test
require_shell_success("Exact Phase-1 IDM test")
PHASE1_IDM_RUN_DIR = backup_pointer_run("idm")
show_json(PHASE1_IDM_RUN_DIR / "eval" / "idm_test_summary.json")

!python -m scripts.compare_runs --phase1
require_shell_success("Fingerprint-checked Phase-1 comparison")
for name in ["phase1_comparison.csv", "phase1_comparison.json", "phase1_comparison.png", "phase1_training_returns.png", "phase1_compare.log"]:
    path = REPO_DIR / "runs" / name
    if path.exists():
        copy_to_drive(path)
result_macros = REPO_DIR / "reports" / "generated_phase1_results.tex"
if result_macros.exists():
    copy_to_drive(result_macros, category="reports")

2026-07-22 04:28:32.669054: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 04:28:32.737732: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
INFO: Run directory: runs/20260722_042836_idm_baseline_seed0
INFO: Evaluating IDMPolicy for 20 episodes on the test split beginning at seed 4000.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cu

## 8. Verify the corrected Phase-1 video API

In [11]:
PHASE1_PPO_RUN_DIR = read_latest_run("ppo")
PHASE1_SAC_RUN_DIR = read_latest_run("sac")
PPO_VIDEO_SEED = representative_seed(PHASE1_PPO_RUN_DIR)
SAC_VIDEO_SEED = representative_seed(PHASE1_SAC_RUN_DIR)
!python -m scripts.record_video --run-dir "{PHASE1_PPO_RUN_DIR}" --model best --seed {PPO_VIDEO_SEED} --steps {VIDEO_STEPS}
require_shell_success("PPO video")
copy_to_drive(PHASE1_PPO_RUN_DIR)
!python -m scripts.record_video --run-dir "{PHASE1_SAC_RUN_DIR}" --model best --seed {SAC_VIDEO_SEED} --steps {VIDEO_STEPS}
require_shell_success("SAC video")
copy_to_drive(PHASE1_SAC_RUN_DIR)

2026-07-22 04:28:59.541887: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 04:28:59.617031: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
INFO: Loading best model on cpu for deterministic scenario seed 4000: /content/safedrive/runs/20260722_012443_

PosixPath('/content/drive/MyDrive/SafeDrive/runs/20260722_012638_sac_phase1_control_sac_seed0')

## 9. Phase-2 task-solvability checks

IDM is informative; ExpertPolicy is the feasibility gate. These use validation scenarios and do not consume the untouched test split.

In [17]:
!python -m scripts.evaluate_baseline --config configs/sac_phase2_direct.yaml --policy idm --split validation --episodes 25 --prefix phase2_idm_validation
require_shell_success("Phase-2 IDM feasibility check")
PHASE2_IDM_VALIDATION_RUN = newest_run_containing("idm_baseline")
copy_to_drive(PHASE2_IDM_VALIDATION_RUN)
show_json(PHASE2_IDM_VALIDATION_RUN / "eval" / "phase2_idm_validation_summary.json")

!python -m scripts.evaluate_baseline --config configs/sac_phase2_direct.yaml --policy expert --split validation --episodes 25 --prefix phase2_expert_validation
require_shell_success("Phase-2 ExpertPolicy feasibility check")
PHASE2_EXPERT_VALIDATION_RUN = newest_run_containing("expert_baseline")
copy_to_drive(PHASE2_EXPERT_VALIDATION_RUN)
expert_summary = show_json(PHASE2_EXPERT_VALIDATION_RUN / "eval" / "phase2_expert_validation_summary.json")
if not require_metrics(expert_summary, "Phase-2 ExpertPolicy feasibility"):
    raise RuntimeError("ExpertPolicy did not solve the target task. Inspect horizon and generated maps before training SAC.")

2026-07-22 04:43:05.425030: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 04:43:05.495152: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
INFO: Run directory: runs/20260722_044309_idm_baseline_seed0
INFO: Evaluating IDMPolicy for 25 episodes on the validation split beginning at seed 20000.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use

## 10. Seed-0 direct SAC pilot

In [21]:
DIRECT_SEED0_RUN = ensure_direct_sac(0)
DIRECT_SEED0_SUMMARY = show_json(DIRECT_SEED0_RUN / "eval" / "best_test_summary.json")

2026-07-22 04:59:42.482270: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 04:59:42.552244: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
INFO: Run directory: runs/20260722_045946_sac_phase2_direct_sac_seed0
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1

## 11. Seed-0 curriculum SAC: curve stage

The large replay buffer is copied to Drive because the next stage resumes it. The lightweight Mac sync will skip it.

In [22]:
CURRICULUM_SEED0_RUN = advance_curriculum_sac(0, "curve")
require_curriculum_state(CURRICULUM_SEED0_RUN, {"paused", "complete"})

2026-07-22 07:14:12.008295: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 07:14:12.078833: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
INFO: Curriculum run directory: runs/20260722_071416_sac_phase2_curriculum_sac_seed0
INFO: Curriculum progress: 0/500000 timesteps; next stage index 0.
INFO: Starting curriculum stage curve (1/3), map='C', maximum 100000 timesteps.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart mod

{'status': 'paused',
 'created_at_utc': '2026-07-22T07:14:16.114362+00:00',
 'updated_at_utc': '2026-07-22T07:32:59.350991+00:00',
 'total_budget': 500000,
 'completed_timesteps': 100000,
 'next_stage_index': 1,
 'stage_order': ['curve', 'straight_curve', 'procedural'],
 'completed_stages': [{'name': 'curve',
   'map': 'C',
   'started_with_total_timesteps': 0,
   'completed_timesteps': 100000,
   'ended_with_total_timesteps': 100000,
   'gate_reached': True,
   'required_gate': True,
   'completed_at_utc': '2026-07-22T07:32:56.552980+00:00',
   'validation_fingerprint': {'schema_version': 2,
    'task_id': '566da43aabbafc06aae61293dc693146555174238c674a39e2f593d31ae67770',
    'strict_id': '2c7b420fafa2b4fad2d67a60026178ca711b941b98a1789cac513271ddb75de1',
    'task': {'environment': {'map': 'C',
      'map_config': None,
      'traffic_density': 0.0,
      'traffic_mode': None,
      'random_traffic': False,
      'random_spawn_lane_index': False,
      'random_lane_num': None,
     

## 11.1 Seed-0 curriculum SAC: straight-plus-curve stage

In [23]:
CURRICULUM_SEED0_RUN = advance_curriculum_sac(0, "straight_curve")
require_curriculum_state(CURRICULUM_SEED0_RUN, {"paused", "complete"})

2026-07-22 07:33:06.566638: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 07:33:06.639277: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
INFO: Curriculum run directory: /content/safedrive/runs/20260722_071416_sac_phase2_curriculum_sac_seed0
INFO: Curriculum progress: 100000/500000 timesteps; next stage index 1.
INFO: Starting curriculum stage straight_curve (2/3), map='SC', maximum 150000 timesteps.
<frozen importlib._bootstrap_external>:1301: 

{'status': 'paused',
 'created_at_utc': '2026-07-22T07:14:16.114362+00:00',
 'updated_at_utc': '2026-07-22T07:38:47.335934+00:00',
 'total_budget': 500000,
 'completed_timesteps': 125000,
 'next_stage_index': 2,
 'stage_order': ['curve', 'straight_curve', 'procedural'],
 'completed_stages': [{'name': 'curve',
   'map': 'C',
   'started_with_total_timesteps': 0,
   'completed_timesteps': 100000,
   'ended_with_total_timesteps': 100000,
   'gate_reached': True,
   'required_gate': True,
   'completed_at_utc': '2026-07-22T07:32:56.552980+00:00',
   'validation_fingerprint': {'schema_version': 2,
    'task_id': '566da43aabbafc06aae61293dc693146555174238c674a39e2f593d31ae67770',
    'strict_id': '2c7b420fafa2b4fad2d67a60026178ca711b941b98a1789cac513271ddb75de1',
    'task': {'environment': {'map': 'C',
      'map_config': None,
      'traffic_density': 0.0,
      'traffic_mode': None,
      'random_traffic': False,
      'random_spawn_lane_index': False,
      'random_lane_num': None,
     

## 11.2 Seed-0 curriculum SAC: procedural target and held-out test

In [24]:
CURRICULUM_SEED0_RUN = advance_curriculum_sac(0, "procedural")
require_curriculum_state(CURRICULUM_SEED0_RUN, {"complete"})
CURRICULUM_SEED0_SUMMARY = show_json(CURRICULUM_SEED0_RUN / "eval" / "best_test_summary.json")

2026-07-22 07:38:53.497508: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 07:38:53.569411: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
INFO: Curriculum run directory: /content/safedrive/runs/20260722_071416_sac_phase2_curriculum_sac_seed0
INFO: Curriculum progress: 125000/500000 timesteps; next stage index 2.
INFO: Starting curriculum stage procedural (3/3), map=3, maximum 375000 timesteps.
<frozen importlib._bootstrap_external>:1301: FutureW

## 12. Compare seed-0 pilots and decide whether confirmation is justified

In [13]:
restore_phase2_session_state()
if PILOTS_PROMISING is None:
    raise RuntimeError("Both seed-0 held-out summaries are required before comparison.")
run_project_module("scripts.compare_runs", ["--phase2", "--seeds", "0"], "seed-0 Phase-2 comparison")
print(f"Confirmation justified: {PILOTS_PROMISING}")
if not PILOTS_PROMISING:
    print("Stop here. Do not spend credits on seeds 1 and 2; inspect the seed-0 diagnostics first.")
for name in ["phase2_seed_results.csv", "phase2_comparison.csv", "phase2_comparison.json", "phase2_comparison.png", "phase2_training_returns.png", "phase2_compare.log"]:
    path = REPO_DIR / "runs" / name
    if path.exists():
        copy_to_drive(path)

INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
INFO: Phase-2 seed results written: runs/phase2_seed_results.csv
INFO: Phase-2 aggregate comparison written: runs/phase2_comparison.csv
INFO: Phase-2 comparison plot written: runs/phase2_comparison.png
INFO: Phase-2 LaTeX result macros written: reports/generated_phase2_results.tex
INFO: Phase-2 training plot written: runs/phase2_training_returns.png


NameError: name 'DIRECT_SEED0_SUMMARY' is not defined

## 13. Conditional confirmation: seed 1

Run this section only when the previous cell prints `Confirmation justified: True`. The curriculum command pauses and backs up at both prerequisite stages inside this cell.

In [26]:
restore_phase2_session_state()
assert PILOTS_PROMISING is True, "Seed-0 pilots did not justify confirmation runs."
DIRECT_SEED1_RUN = ensure_direct_sac(1)
CURRICULUM_SEED1_RUN = advance_curriculum_sac(1, "procedural")

2026-07-22 07:47:43.410719: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 07:47:43.483660: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
INFO: Run directory: runs/20260722_074747_sac_phase2_direct_sac_seed1
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1

PosixPath('/content/drive/MyDrive/SafeDrive/runs/20260722_092850_sac_phase2_curriculum_sac_seed1')

## 13.1 Conditional confirmation: seed 2

This cell is restart-safe. A completed direct run is reused, and a curriculum `failed_gate` result is reported as a terminal experimental outcome rather than retrained or forced into later stages.

In [12]:
restore_phase2_session_state()
assert PILOTS_PROMISING is True, "Seed-0 pilots did not justify confirmation runs."
DIRECT_SEED2_RUN = ensure_direct_sac(2)
CURRICULUM_SEED2_RUN = advance_curriculum_sac(2, "procedural")
if learned_run_status(CURRICULUM_SEED2_RUN) == "failed_gate":
    print("Seed 2 is intentionally excluded from paired comparison; do not rerun it.")

NameError: name 'PILOTS_PROMISING' is not defined

## 14. Exact Phase-2 IDM and ExpertPolicy test baselines

In [ ]:
PHASE2_IDM_RUN = ensure_baseline("idm", "phase2_idm", "phase2_idm_test")
PHASE2_EXPERT_RUN = ensure_baseline("expert", "phase2_expert", "phase2_expert_test")

## 15. Zero-shot light-traffic stress test

In [ ]:
restore_phase2_session_state()
ensure_evaluation(DIRECT_SEED0_RUN, prefix="best_light_traffic", overrides=["test.traffic_density=0.05", "test.random_traffic=true"])
ensure_evaluation(CURRICULUM_SEED0_RUN, prefix="best_light_traffic", overrides=["test.traffic_density=0.05", "test.random_traffic=true"])
if PILOTS_PROMISING:
    print(f"Run the next optional cell for complete confirmation pairs: {[seed for seed in COMPLETED_PHASE2_SEEDS if seed != 0]}")

### Optional confirmation light-traffic evaluations

Only complete direct/curriculum seed pairs are evaluated. Incomplete or failed-gate seeds are excluded automatically.

In [ ]:
restore_phase2_session_state()
assert PILOTS_PROMISING is True, "Confirmation runs were not justified."
for seed in [seed for seed in COMPLETED_PHASE2_SEEDS if seed != 0]:
    direct_run = globals()[f"DIRECT_SEED{seed}_RUN"]
    curriculum_run = globals()[f"CURRICULUM_SEED{seed}_RUN"]
    ensure_evaluation(direct_run, prefix="best_light_traffic", overrides=["test.traffic_density=0.05", "test.random_traffic=true"])
    ensure_evaluation(curriculum_run, prefix="best_light_traffic", overrides=["test.traffic_density=0.05", "test.random_traffic=true"])

## 16. Final strict comparison and representative videos

In [ ]:
restore_phase2_session_state()
COMPARISON_SEEDS = COMPLETED_PHASE2_SEEDS
if not COMPARISON_SEEDS:
    raise RuntimeError("No complete direct/curriculum seed pairs are available.")
subprocess.run([sys.executable, "-m", "scripts.compare_runs", "--phase2", "--seeds", *[str(seed) for seed in COMPARISON_SEEDS]], cwd=REPO_DIR, check=True)
for path in (REPO_DIR / "runs").glob("phase2_*"):
    if path.is_file():
        copy_to_drive(path)

DIRECT_VIDEO_SEED = representative_seed(DIRECT_SEED0_RUN)
CURRICULUM_VIDEO_SEED = representative_seed(CURRICULUM_SEED0_RUN)
ensure_video(DIRECT_SEED0_RUN, DIRECT_VIDEO_SEED)
ensure_video(CURRICULUM_SEED0_RUN, CURRICULUM_VIDEO_SEED)

## 17. Build reports and perform final artifact sync

In [ ]:
if shutil.which("latexmk"):
    subprocess.run(["latexmk", "-cd", "-pdf", "-interaction=nonstopmode", "-halt-on-error", "reports/main.tex"], cwd=REPO_DIR, check=True)
    subprocess.run(["latexmk", "-cd", "-pdf", "-interaction=nonstopmode", "-halt-on-error", "reports/surrogate_notes.tex"], cwd=REPO_DIR, check=True)
else:
    print("latexmk is unavailable in this runtime; compile the reports locally.")
(DRIVE_PROJECT / "runs").mkdir(parents=True, exist_ok=True)
(DRIVE_PROJECT / "reports").mkdir(parents=True, exist_ok=True)
if shutil.which("rsync"):
    subprocess.run(["rsync", "-a", f"{REPO_DIR / 'runs'}/", f"{DRIVE_PROJECT / 'runs'}/"], check=True)
    subprocess.run(["rsync", "-a", f"{REPO_DIR / 'reports'}/", f"{DRIVE_PROJECT / 'reports'}/"], check=True)
else:
    shutil.copytree(REPO_DIR / "runs", DRIVE_PROJECT / "runs", dirs_exist_ok=True)
    shutil.copytree(REPO_DIR / "reports", DRIVE_PROJECT / "reports", dirs_exist_ok=True)
print(f"Final artifacts synchronized to {DRIVE_PROJECT}")

## Finished

Disconnect the Colab runtime when complete. On the Mac, run `python -m scripts.sync_drive_runs` to bring back only analysis artifacts, logs, plots, summaries, videos, and reports.